# CHAPTER 8 : Data Wrangling: Join, Combine, and Reshape

-----

In many applications, data may be spread across a number of files or databases or be
arranged in a form that is not easy to analyze. This chapter focuses on tools to help
combine, join, and rearrange data.

First, I introduce the concept of hierarchical indexing in pandas, which is used extensively
in some of these operations. I then dig into the particular data manipulations.
You can see various applied usages of these tools in Chapter 14.

## Basic Imports and Set ups

In [1]:
import pandas as pd
import numpy as np
import os
from faker import Faker

This is a function to render dataframes as tables in the Jupyter Notebook.

In [2]:
import pandas as pd
from IPython.display import display, HTML

def render_book_table(title, columns, rows):
    """
    Render a documentation-style table in Jupyter.

    Parameters
    ----------
    title : str
        Title displayed above the table
    columns : list
        Column names
    rows : list of lists
        Table data rows
    """

    table_data = pd.DataFrame(rows, columns=columns)

    styled_table = (
        table_data.style
        .hide(axis="index")
        .set_table_styles([
            {"selector": "th",
             "props": [("background-color", "#f2f2f2"),
                       ("font-weight", "bold"),
                       ("text-align", "center"),
                       ("border", "1px solid #999"),
                       ("padding", "8px")]},

            {"selector": "td",
             "props": [("border", "1px solid #999"),
                       ("padding", "8px"),
                       ("white-space", "normal"),
                       ("text-align", "left")]},

            {"selector": "table",
             "props": [("border-collapse", "collapse"),
                       ("width", "100%")]}
        ])
    )

    display(HTML(f"<h3>{title}</h3>"))
    display(styled_table)



## 8.1 Hierarchical Indexing

*Hierarchical indexing* is an important feature of pandas that enables you to have multiple
(two or more) index levels on an axis. Somewhat abstractly, it provides a way for
you to work with higher dimensional data in a lower dimensional form. Let’s start
with a simple example; create a Series with a list of lists (or arrays) as the index:

In [6]:
data = pd.Series(np.random.randn(9),index=[['a', 'a', 'a', 'b', 'b', 'c', 'c', 'd', 'd'],[1, 2, 3, 1, 3, 1, 2, 2, 3]])

In [7]:
data # You can see 2 indexes for Series.

a  1   -0.688790
   2    0.793909
   3    0.873270
b  1    1.889498
   3   -0.951723
c  1   -0.149447
   2   -0.152598
d  2    0.474364
   3   -0.680989
dtype: float64

What you’re seeing is a prettified view of a Series with a MultiIndex as its index. The
“gaps” in the index display mean “use the label directly above”:

In [8]:
data.index

MultiIndex([('a', 1),
            ('a', 2),
            ('a', 3),
            ('b', 1),
            ('b', 3),
            ('c', 1),
            ('c', 2),
            ('d', 2),
            ('d', 3)],
           )

In [13]:
data.index.levels

FrozenList([['a', 'b', 'c', 'd'], [1, 2, 3]])

In [12]:
data.index.codes  # Codes was earlier called Labels.

FrozenList([[0, 0, 0, 1, 1, 2, 2, 3, 3], [0, 1, 2, 0, 2, 0, 1, 1, 2]])

With a hierarchically indexed object, so-called partial indexing is possible, enabling
you to concisely select subsets of the data

In [9]:
data['b']

1    1.889498
3   -0.951723
dtype: float64

---------

##### ✅ What does **`labels`** mean in this `MultiIndex`?

When you create a **`MultiIndex`**, pandas internally stores it in a **factorized (encoded) form** to save memory and improve performance.

It does **NOT store the actual index values repeatedly.**
Instead, it stores:

* **`levels` → the unique values at each index level**
* **`labels` → integer pointers (positions) telling which level value to use for each row**

---

##### 🔵 In your example

You created:

```python
data = pd.Series(
    np.random.randn(9),
    index=[
        ['a', 'a', 'a', 'b', 'b', 'c', 'c', 'd', 'd'],
        [1, 2, 3, 1, 3, 1, 2, 2, 3]
    ]
)
```

And pandas shows:

```text
MultiIndex(
  levels=[['a','b','c','d'], [1,2,3]],
  labels=[[0,0,0,1,1,2,2,3,3], [0,1,2,0,2,0,1,1,2]]
)
```

---

##### 🟢 Step-1: Understand `levels`

These are the **unique sorted values** for each index level.

Level-0:

```
['a','b','c','d']
```

Level-1:

```
[1,2,3]
```

---

##### 🟢 Step-2: Understand `labels`

`labels` are **integer arrays that reference positions inside the `levels` list.**

Think of them as:

> “Which value from `levels` should be used at this row?”

---

###### ⭐ Level-0 labels

```
[0,0,0,1,1,2,2,3,3]
```

Mapping using:

```
levels[0] = ['a','b','c','d']
```

So:

| Label | Means |
| ----- | ----- |
| 0 →   | 'a'   |
| 1 →   | 'b'   |
| 2 →   | 'c'   |
| 3 →   | 'd'   |

Thus pandas reconstructs:

```
a a a b b c c d d
```

---

###### ⭐ Level-1 labels

```
[0,1,2,0,2,0,1,1,2]
```

Mapping using:

```
levels[1] = [1,2,3]
```

So:

| Label | Means |
| ----- | ----- |
| 0 →   | 1     |
| 1 →   | 2     |
| 2 →   | 3     |

Thus pandas reconstructs:

```
1 2 3 1 3 1 2 2 3
```

---

##### 🎯 Final Reconstruction Logic

For row `i`:

```
actual_index_value =
(
   levels[0][ labels[0][i] ],
   levels[1][ labels[1][i] ]
)
```

Example → row 4

```
labels → (1 , 2)

levels[0][1] → 'b'
levels[1][2] → 3

→ index = ('b', 3)
```

---

##### 🚀 Why pandas does this (VERY important Data Science insight)

This is basically **categorical / dictionary encoding of index values.**

Benefits:

✅ avoids storing repeated strings
✅ faster grouping / sorting
✅ lower memory
✅ vectorized operations on integer codes

This exact idea appears in:

* pandas `Categorical`
* database dictionary encoding
* parquet column encoding
* feature engineering pipelines

---

##### ⚠️ Modern pandas note (important)

In **new pandas versions**, `labels` is renamed to:

```
codes
```

So you may see:

```python
data.index.codes
```

instead of:

```
data.index.labels
```

Same concept. Only name changed.

---



With a hierarchically indexed object, so-called partial indexing is possible, enabling
you to concisely select subsets of the data:

In [14]:
data['b']

1    1.889498
3   -0.951723
dtype: float64

In [15]:
data['b':'c']

b  1    1.889498
   3   -0.951723
c  1   -0.149447
   2   -0.152598
dtype: float64

In [16]:
data.loc[['b', 'd']]

b  1    1.889498
   3   -0.951723
d  2    0.474364
   3   -0.680989
dtype: float64

Selection is even possible from an “inner” level:

In [18]:
data.loc[:, 2]

a    0.793909
c   -0.152598
d    0.474364
dtype: float64

Hierarchical indexing plays an important role in reshaping data and group-based
operations like forming a pivot table. For example, you could rearrange the data into
a DataFrame using its `unstack` method:

In [19]:
data.unstack()

,1,2,3
a,-0.688790,0.793909,0.873270
b,1.889498,NaN,-0.951723
c,-0.149447,-0.152598,NaN
d,NaN,0.474364,-0.680989


The inverse operation of `unstack` is `stack`:

In [21]:
data.unstack().stack()

a  1   -0.688790
   2    0.793909
   3    0.873270
b  1    1.889498
   3   -0.951723
c  1   -0.149447
   2   -0.152598
d  2    0.474364
   3   -0.680989
dtype: float64

`stack` and `unstack`will be explored in more detail later in this chapter.

----

In [55]:
frame = pd.DataFrame(np.arange(12).reshape((4, 3)),index=[['a', 'a', 'b', 'b'], [1, 2, 1, 2]],columns=[['Ohio', 'Ohio', 'Colorado'],['Green', 'Red', 'Green']])

With a DataFrame, either axis can have a hierarchical index:

In [56]:
frame

Ohio     Colorado
    Green Red    Green
a 1     0   1        2
  2     3   4        5
b 1     6   7        8
  2     9  10       11

The hierarchical levels can have names (as strings or any Python objects). If so, these
will show up in the console output:

In [57]:
frame.index.names = ['key1', 'key2']

In [58]:
frame.columns.names = ['state', 'color']

In [59]:
frame

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

> Be careful to distinguish the index names 'state' and 'color'
from the row labels.

With partial column indexing you can similarly select groups of columns:

In [60]:
frame['Ohio']

color      Green  Red
key1 key2            
a    1         0    1
     2         3    4
b    1         6    7
     2         9   10

A MultiIndex can be created by itself and then reused; the columns in the preceding
DataFrame with level names could be created like this:

In [63]:
mindex_for_cols = pd.MultiIndex.from_arrays([['Ohio', 'Ohio', 'Colorado'], ['Green', 'Red', 'Green']],
names=['state', 'color'])

In [64]:
mindex_for_rows = pd.MultiIndex.from_arrays([['a', 'a', 'b', 'b'], [1, 2, 1, 2]],
names=['key1', 'key2'])

In [68]:
frame2 = pd.DataFrame(np.arange(12).reshape((4, 3)),index= mindex_for_rows,columns=mindex_for_cols)

In [70]:
frame2

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

### Reordering and Sorting Levels

At times you will need to rearrange the order of the levels on an axis or sort the data
by the values in one specific level. The `swaplevel` takes two level numbers or names
and returns a new object with the levels interchanged (but the data is otherwise
unaltered):

In [71]:
frame

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

In [72]:
frame.swaplevel('key1', 'key2')

state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
2    a        3   4        5
1    b        6   7        8
2    b        9  10       11

`sort_index`, on the other hand, sorts the data using only the values in a single level.
When swapping levels, it’s not uncommon to also use `sort_index` so that the result is
lexicographically sorted by the indicated level:

In [74]:
frame.sort_index(level=1)

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
b    1        6   7        8
a    2        3   4        5
b    2        9  10       11

In [76]:
frame.sort_index(level=0,ascending= False)

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
b    2        9  10       11
     1        6   7        8
a    2        3   4        5
     1        0   1        2

In [77]:
frame.sort_index(level=1, ascending= False)

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
b    2        9  10       11
a    2        3   4        5
b    1        6   7        8
a    1        0   1        2

In [78]:
frame.swaplevel(0, 1).sort_index(level=0)

state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
     b        6   7        8
2    a        3   4        5
     b        9  10       11

> Data selection performance is much better on hierarchically
indexed objects if the index is lexicographically sorted starting with
the outermost level—that is, the result of calling
`sort_index(level=0)` or `sort_index()`.

### Summary Statistics by Level

Many descriptive and summary statistics on DataFrame and Series have a level
option in which you can specify the level you want to aggregate by on a particular
axis. Consider the above DataFrame; we can aggregate by level on either the rows or
columns like so:

In [79]:
frame

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

In [80]:
frame.sum(level='key2')

TypeError: sum() got an unexpected keyword argument 'level'

This is failing because in newer version as of the time of this notebook, this option is deprecated. 

##### ✅ Why you are getting this error

You are seeing:

```
TypeError: sum() got an unexpected keyword argument 'level'
```

because **in modern pandas versions (`>= 1.3` and fully removed later)** the `level=` argument in reduction functions like:

```
frame.sum(level='key2')
```

has been **deprecated and removed.**

Earlier (very old pandas — like in the book/course you are studying), this worked because pandas allowed:

```
DataFrame.sum(level=...)
```

to perform aggregation across a **MultiIndex level.**

That API is now replaced with **`groupby(level=...)`.**

---

##### ✅ Correct Modern Pandas Way (Very Important)

You must now write:

```
frame2.groupby(level='key2').sum()
```

This is the **exact modern equivalent** of:

```
frame2.sum(level='key2')
```

---

##### ✅ Let’s Understand What is Happening Conceptually

Your dataframe:

* Rows → MultiIndex (`key1`, `key2`)
* Columns → MultiIndex (`state`, `color`)

When you do:

```
groupby(level='key2')
```

You are saying:

👉 **“Group rows that have the same `key2` value and then sum them.”**

So rows:

```
(a,1)
(b,1)
```

will be grouped together

and rows:

```
(a,2)
(b,2)
```

will be grouped together

Then sum is computed column-wise.

---

##### ✅ Full Working Example

```
frame2.groupby(level='key2').sum()
```

Result structure:

* Index → only `key2`
* Columns → unchanged MultiIndex columns

---

##### ✅ If You Wanted Column Level Aggregation

Very powerful point (senior-level understanding 🙂)

If you wanted to aggregate **across column MultiIndex level**, you would do:

```
frame2.groupby(level='state', axis=1).sum()
```

So now pandas groups **columns instead of rows.**

---

##### ✅ Why Pandas Removed `level=` from `.sum()`

Architectural reason:

* Old API → hidden grouping behaviour inside reduction
* New API → **explicit grouping step**
* Cleaner mental model
* Consistent with:

```
groupby → aggregate
```

This aligns pandas more with SQL / distributed systems / dataframe engines.

---

##### ⭐ Senior Insight (Very Useful for You in Data Science)

Whenever you see:

```
something.sum(level=...)
```

Immediately translate in your head to:

```
something.groupby(level=...).sum()
```

This will save you **huge debugging time** when reading older books / notebooks / courses.

---


In [81]:
frame2.groupby(level='key2').sum()

state  Ohio     Colorado
color Green Red    Green
key2                    
1         6   8       10
2        12  14       16

Under the hood, this utilizes pandas’s groupby machinery, which will be discussed in
more detail later in the book.

### Indexing with a DataFrame’s columns

It’s not unusual to want to use one or more columns from a DataFrame as the row
index; alternatively, you may wish to move the row index into the DataFrame’s columns.
Here’s an example DataFrame:

In [84]:
frame = pd.DataFrame({'a': range(7), 'b': range(7, 0, -1),
                      'c': ['one', 'one', 'one', 'two', 'two',
                            'two', 'two'],
                      'd': [0, 1, 2, 0, 1, 2, 3]})

In [85]:
frame

,a,b,c,d
0,0,7,one,0
1,1,6,one,1
2,2,5,one,2
3,3,4,two,0
4,4,3,two,1
5,5,2,two,2
6,6,1,two,3


DataFrame’s `set_index` function will create a new DataFrame using one or more of
its columns as the index:

In [86]:
frame2 = frame.set_index(['c', 'd'])

In [87]:
frame2

a  b
c   d      
one 0  0  7
    1  1  6
    2  2  5
two 0  3  4
    1  4  3
    2  5  2
    3  6  1

By default the columns are removed from the DataFrame, though you can leave them
in:

In [88]:
frame.set_index(['c', 'd'], drop=False)

a  b    c  d
c   d              
one 0  0  7  one  0
    1  1  6  one  1
    2  2  5  one  2
two 0  3  4  two  0
    1  4  3  two  1
    2  5  2  two  2
    3  6  1  two  3

`reset_index`, on the other hand, does the opposite of `set_index`; the hierarchical
index levels are moved into the columns:



---

## 8.2 Combining and Merging Datasets

Data contained in pandas objects can be combined together in a number of ways:

• `pandas.merge` connects rows in DataFrames based on one or more keys. This
will be familiar to users of SQL or other relational databases, as it implements
database join operations.

• `pandas.concat` concatenates or “stacks” together objects along an axis.

• The `combine_first` instance method enables splicing together overlapping data
to fill in missing values in one object with values from another.

I will address each of these and give a number of examples. They’ll be utilized in
examples throughout the rest of the book.

### Database-Style DataFrame Joins

*Merge* or *join* operations combine datasets by linking rows using one or more keys.
These operations are central to relational databases (e.g., SQL-based). The `merge`
function in pandas is the main entry point for using these algorithms on your data.

Let’s start with a simple example:

In [100]:
df1 = pd.DataFrame({'key': ['b', 'b', 'a', 'c', 'a', 'a', 'b'],'data1': range(7)})

In [101]:
df2 = pd.DataFrame({'key': ['a', 'b', 'd'],'data2': range(3)})

In [102]:
df1

,key,data1
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,a,5
6,b,6


In [103]:
df2

,key,data2
0,a,0
1,b,1
2,d,2


This is an example of a many-to-one join; the data in `df1` has multiple rows labeled a
and b, whereas `df2` has only one row for each value in the key column. Calling `merge`
with these objects we obtain:

In [105]:
pd.merge(df1, df2)

,key,data1,data2
0,b,0,1
1,b,1,1
2,a,2,0
3,a,4,0
4,a,5,0
5,b,6,1


Note that I didn’t specify which column to join on. If that information is not specified,
**merge uses the overlapping column names as the keys**. It’s a good practice to
specify explicitly, though:

In [107]:
pd.merge(df1, df2, on='key')

,key,data1,data2
0,b,0,1
1,b,1,1
2,a,2,0
3,a,4,0
4,a,5,0
5,b,6,1


If the column names are different in each object, you can specify them separately:

In [108]:
df3 = pd.DataFrame({'lkey': ['b', 'b', 'a', 'c', 'a', 'a', 'b'],'data1': range(7)})

In [109]:
df4 = pd.DataFrame({'rkey': ['a', 'b', 'd'],'data2': range(3)})

In [110]:
pd.merge(df3, df4, left_on='lkey', right_on='rkey')

,lkey,data1,rkey,data2
0,b,0,b,1
1,b,1,b,1
2,a,2,a,0
3,a,4,a,0
4,a,5,a,0
5,b,6,b,1


You may notice that the `'c'` and `'d'` values and associated data are missing from the
result. By default merge does an `'inner'` join; the keys in the result are the intersection,
or the common set found in both tables. Other possible options are `'left'`,
`'right'`, and `'outer'`. The outer join takes the union of the keys, combining the effect of applying both left and right joins:

In [111]:
pd.merge(df1, df2, how='outer')

,key,data1,data2
0,a,2.0,0.0
1,a,4.0,0.0
2,a,5.0,0.0
3,b,0.0,1.0
4,b,1.0,1.0
5,b,6.0,1.0
6,c,3.0,NaN
7,d,NaN,2.0


See Table 8-1 for a summary of the options for how.

In [113]:
columns = ["Option", "Behavior"]

rows = [
["inner","Use only the key combinations observed in both tables"],
["left","Use all key combinations found in the left table"],
["right","Use all key combinations found in the right table"],
["outer","Use all key combinations observed in both tables together"]
]

render_book_table(
    "Table 8-1. Different Join Types with how Argument",
    columns,
    rows
)

Option,Behavior
inner,Use only the key combinations observed in both tables
left,Use all key combinations found in the left table
right,Use all key combinations found in the right table
outer,Use all key combinations observed in both tables together


Many-to-many merges have well-defined, though not necessarily intuitive, behavior.
Here’s an example:

In [116]:
df1 = pd.DataFrame({'key': ['b', 'b', 'a', 'c', 'a', 'b'],'data1': range(6)})

In [117]:
df2 = pd.DataFrame({'key': ['a', 'b', 'a', 'b', 'd'],'data2': range(5)})

In [118]:
df1

,key,data1
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,b,5


In [119]:
df2

,key,data2
0,a,0
1,b,1
2,a,2
3,b,3
4,d,4


In [120]:
pd.merge(df1, df2, on='key', how='left')

,key,data1,data2
0,b,0,1.0
1,b,0,3.0
2,b,1,1.0
3,b,1,3.0
4,a,2,0.0
5,a,2,2.0
6,c,3,NaN
7,a,4,0.0
8,a,4,2.0
9,b,5,1.0


Many-to-many joins form the Cartesian product of the rows. Since there were three
`'b'` rows in the left DataFrame and two in the right one, there are six `'b'` rows in the
result. The join method only affects the distinct key values appearing in the result:

In [121]:
pd.merge(df1, df2, how='inner')

,key,data1,data2
0,b,0,1
1,b,0,3
2,b,1,1
3,b,1,3
4,a,2,0
5,a,2,2
6,a,4,0
7,a,4,2
8,b,5,1
9,b,5,3


----

To merge with multiple keys, pass a list of column names:

In [123]:
left = pd.DataFrame({'key1': ['foo', 'foo', 'bar'],'key2': ['one', 'two', 'one'],'lval': [1, 2, 3]})

In [124]:
right = pd.DataFrame({'key1': ['foo', 'foo', 'bar', 'bar'],'key2': ['one', 'one', 'one', 'two'],'rval': [4, 5, 6, 7]})

In [125]:
left

,key1,key2,lval
0,foo,one,1
1,foo,two,2
2,bar,one,3


In [126]:
right

,key1,key2,rval
0,foo,one,4
1,foo,one,5
2,bar,one,6
3,bar,two,7


In [127]:
pd.merge(left, right, on=['key1', 'key2'], how='outer')

,key1,key2,lval,rval
0,bar,one,3.0,6.0
1,bar,two,NaN,7.0
2,foo,one,1.0,4.0
3,foo,one,1.0,5.0
4,foo,two,2.0,NaN


To determine which key combinations will appear in the result depending on the
choice of merge method, think of the multiple keys as forming an array of tuples to
be used as a single join key (even though it’s not actually implemented that way).

> When you’re joining columns-on-columns, the indexes on the
passed DataFrame objects are discarded.

A last issue to consider in merge operations is the treatment of overlapping column
names. While you can address the overlap manually (see the earlier section on
renaming axis labels), `merge` has a `suffixes` option for specifying strings to append
to overlapping names in the left and right DataFrame objects:

In [128]:
pd.merge(left, right, on='key1')

,key1,key2_x,lval,key2_y,rval
0,foo,one,1,one,4
1,foo,one,1,one,5
2,foo,two,2,one,4
3,foo,two,2,one,5
4,bar,one,3,one,6
5,bar,one,3,two,7


In [131]:
pd.merge(left, right, on='key2', how='outer')

,key1_x,key2,lval,key1_y,rval
0,foo,one,1,foo,4
1,foo,one,1,foo,5
2,foo,one,1,bar,6
3,bar,one,3,foo,4
4,bar,one,3,foo,5
5,bar,one,3,bar,6
6,foo,two,2,bar,7


In [132]:
pd.merge(left, right, on='key1', suffixes=('_left', '_right'))

,key1,key2_left,lval,key2_right,rval
0,foo,one,1,one,4
1,foo,one,1,one,5
2,foo,two,2,one,4
3,foo,two,2,one,5
4,bar,one,3,one,6
5,bar,one,3,two,7


See Table 8-2 for an argument reference on `merge`. Joining using the DataFrame’s row
index is the subject of the next section.



In [135]:
columns = ["Argument", "Description"]

rows = [
["left","DataFrame to be merged on the left side"],
["right","DataFrame to be merged on the right side"],
["how","One of 'inner', 'outer', 'left', or 'right'; defaults to 'inner'"],
["on","Column names to join on; must exist in both DataFrames"],
["left_on","Columns in left DataFrame to use as join keys"],
["right_on","Analogous to left_on for right DataFrame"],
["left_index","Use row index in left as join key (or keys if MultiIndex)"],
["right_index","Analogous to left_index"],
["sort","Sort merged data lexicographically by join keys; True by default"],
["suffixes","Tuple of suffixes for overlapping column names; default ('_x','_y')"],
["copy","If False, avoid copying data in some exceptional cases"],
["indicator","Adds column '_merge' showing row origin: 'left_only', 'right_only', or 'both'"]
]

render_book_table(
    "Table 8-2. merge Function Arguments",
    columns,
    rows
)

Argument,Description
left,DataFrame to be merged on the left side
right,DataFrame to be merged on the right side
how,"One of 'inner', 'outer', 'left', or 'right'; defaults to 'inner'"
on,Column names to join on; must exist in both DataFrames
left_on,Columns in left DataFrame to use as join keys
right_on,Analogous to left_on for right DataFrame
left_index,Use row index in left as join key (or keys if MultiIndex)
right_index,Analogous to left_index
sort,Sort merged data lexicographically by join keys; True by default
suffixes,"Tuple of suffixes for overlapping column names; default ('_x','_y')"


### Merging on Index

In some cases, the merge key(s) in a DataFrame will be found in its index. In this
case, you can pass `left_index=True` or `right_index=True` (or both) to indicate that
the index should be used as the merge key:

In [138]:
left1 = pd.DataFrame({'key': ['a', 'b', 'a', 'a', 'b', 'c'],'value': range(6)})

In [139]:
right1 = pd.DataFrame({'group_val': [3.5, 7]}, index=['a', 'b'])

In [140]:
left1

,key,value
0,a,0
1,b,1
2,a,2
3,a,3
4,b,4
5,c,5


In [141]:
right1

,group_val
a,3.5
b,7.0


In [142]:
pd.merge(left1, right1, left_on='key', right_index=True)

,key,value,group_val
0,a,0,3.5
1,b,1,7.0
2,a,2,3.5
3,a,3,3.5
4,b,4,7.0


Since the default merge method is to intersect the join keys, you can instead form the
union of them with an outer join:

In [143]:
pd.merge(left1, right1, left_on='key', right_index=True, how='outer')

,key,value,group_val
0,a,0,3.5
2,a,2,3.5
3,a,3,3.5
1,b,1,7.0
4,b,4,7.0
5,c,5,NaN


----

With hierarchically indexed data, things are more complicated, as joining on index is
implicitly a multiple-key merge:

In [145]:
lefth = pd.DataFrame({'key1': ['Ohio', 'Ohio', 'Ohio','Nevada', 'Nevada'],
                      'key2': [2000, 2001, 2002, 2001, 2002],
                      'data': np.arange(5.)})
                      

In [146]:
righth = pd.DataFrame(np.arange(12).reshape((6, 2)),
                      index=[['Nevada', 'Nevada', 'Ohio', 'Ohio','Ohio', 'Ohio'],
                             [2001, 2000, 2000, 2000, 2001, 2002]],
                      columns=['event1', 'event2'])

In [147]:
lefth

,key1,key2,data
0,Ohio,2000,0.0
1,Ohio,2001,1.0
2,Ohio,2002,2.0
3,Nevada,2001,3.0
4,Nevada,2002,4.0


In [149]:
righth

event1  event2
Nevada 2001       0       1
       2000       2       3
Ohio   2000       4       5
       2000       6       7
       2001       8       9
       2002      10      11

In this case, you have to indicate multiple columns to merge on as a list (note the
handling of duplicate index values with `how='outer'`):

In [151]:
pd.merge(lefth, righth, left_on=['key1', 'key2'], right_index=True)

,key1,key2,data,event1,event2
0,Ohio,2000,0.0,4,5
0,Ohio,2000,0.0,6,7
1,Ohio,2001,1.0,8,9
2,Ohio,2002,2.0,10,11
3,Nevada,2001,3.0,0,1


In [152]:
pd.merge(lefth, righth, left_on=['key1', 'key2'],right_index=True, how='outer')

,key1,key2,data,event1,event2
4,Nevada,2000,NaN,2.0,3.0
3,Nevada,2001,3.0,0.0,1.0
4,Nevada,2002,4.0,NaN,NaN
0,Ohio,2000,0.0,4.0,5.0
0,Ohio,2000,0.0,6.0,7.0
1,Ohio,2001,1.0,8.0,9.0
2,Ohio,2002,2.0,10.0,11.0


Using the indexes of both sides of the merge is also possible:

In [155]:
left2 = pd.DataFrame([[1., 2.], [3., 4.], [5., 6.]],
                     index=['a', 'c', 'e'],
                     columns=['Ohio', 'Nevada'])

In [156]:
right2 = pd.DataFrame([[7., 8.], [9., 10.], [11., 12.], [13, 14]],
                      index=['b', 'c', 'd', 'e'],
                      columns=['Missouri', 'Alabama'])

In [157]:
left2

,Ohio,Nevada
a,1.0,2.0
c,3.0,4.0
e,5.0,6.0


In [158]:
right2

,Missouri,Alabama
b,7.0,8.0
c,9.0,10.0
d,11.0,12.0
e,13.0,14.0


In [159]:
pd.merge(left2, right2, how='outer', left_index=True, right_index=True)

,Ohio,Nevada,Missouri,Alabama
a,1.0,2.0,NaN,NaN
b,NaN,NaN,7.0,8.0
c,3.0,4.0,9.0,10.0
d,NaN,NaN,11.0,12.0
e,5.0,6.0,13.0,14.0


DataFrame has a convenient `join` instance for merging by index. It can also be used
to combine together many DataFrame objects having the same or similar indexes but
non-overlapping columns. In the prior example, we could have written:


In [162]:
left2.join(right2, how='outer')

,Ohio,Nevada,Missouri,Alabama
a,1.0,2.0,NaN,NaN
b,NaN,NaN,7.0,8.0
c,3.0,4.0,9.0,10.0
d,NaN,NaN,11.0,12.0
e,5.0,6.0,13.0,14.0


In part for legacy reasons (i.e., much earlier versions of pandas), DataFrame’s `join`
method performs a left join on the join keys, exactly preserving the left frame’s row
index. It also supports joining the index of the passed DataFrame on one of the columns
of the calling DataFrame:

In [163]:
left1

,key,value
0,a,0
1,b,1
2,a,2
3,a,3
4,b,4
5,c,5


In [164]:
right1

,group_val
a,3.5
b,7.0


In [165]:
left1.join(right1, on='key')

,key,value,group_val
0,a,0,3.5
1,b,1,7.0
2,a,2,3.5
3,a,3,3.5
4,b,4,7.0
5,c,5,NaN


Lastly, for simple index-on-index merges, you can pass a list of DataFrames to join as
an alternative to using the more general`concat` function described in the next
section:

In [166]:
another = pd.DataFrame([[7., 8.], [9., 10.], [11., 12.], [16., 17.]],
                       index=['a', 'c', 'e', 'f'],
                       columns=['New York', 'Oregon'])

In [167]:
another

,New York,Oregon
a,7.0,8.0
c,9.0,10.0
e,11.0,12.0
f,16.0,17.0


In [168]:
left2

,Ohio,Nevada
a,1.0,2.0
c,3.0,4.0
e,5.0,6.0


In [169]:
right2

,Missouri,Alabama
b,7.0,8.0
c,9.0,10.0
d,11.0,12.0
e,13.0,14.0


In [170]:
left2.join([right2, another]) # Here only left2 has left join and from rest only common index values taken.

,Ohio,Nevada,Missouri,Alabama,New York,Oregon
a,1.0,2.0,NaN,NaN,7.0,8.0
c,3.0,4.0,9.0,10.0,9.0,10.0
e,5.0,6.0,13.0,14.0,11.0,12.0


In [171]:
left2.join([right2, another], how='outer')

,Ohio,Nevada,Missouri,Alabama,New York,Oregon
a,1.0,2.0,NaN,NaN,7.0,8.0
c,3.0,4.0,9.0,10.0,9.0,10.0
e,5.0,6.0,13.0,14.0,11.0,12.0
b,NaN,NaN,7.0,8.0,NaN,NaN
d,NaN,NaN,11.0,12.0,NaN,NaN
f,NaN,NaN,NaN,NaN,16.0,17.0


### Concatenating Along an Axis

Another kind of data combination operation is referred to interchangeably as concatenation,
binding, or stacking. NumPy’s `concatenate` function can do this with
NumPy arrays

In [173]:
arr = np.arange(12).reshape((3, 4))

In [174]:
arr

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11]])

In [175]:
np.concatenate([arr, arr], axis=1)

array([[ 0,  1,  2,  3,  0,  1,  2,  3],
       [ 4,  5,  6,  7,  4,  5,  6,  7],
       [ 8,  9, 10, 11,  8,  9, 10, 11]])

In the context of pandas objects such as Series and DataFrame, having labeled axes
enable you to further generalize array concatenation. In particular, you have a number
of additional things to think about:

• If the objects are indexed differently on the other axes, should we combine the
distinct elements in these axes or use only the shared values (the intersection)?

• Do the concatenated chunks of data need to be identifiable in the resulting
object?

• Does the “concatenation axis” contain data that needs to be preserved? In many
cases, the default integer labels in a DataFrame are best discarded during
concatenation.

The `concat` function in pandas provides a consistent way to address each of these
concerns. I’ll give a number of examples to illustrate how it works. Suppose we have
three Series with no index overlap:

In [4]:
s1 = pd.Series([0, 1], index=['a', 'b'])

In [5]:
s2 = pd.Series([2, 3, 4], index=['c', 'd', 'e'])

In [6]:
s3 = pd.Series([5, 6], index=['f', 'g'])

Calling `concat` with these objects in a list glues together the values and indexes:

In [7]:
pd.concat([s1, s2, s3])

a    0
b    1
c    2
d    3
e    4
f    5
g    6
dtype: int64

what if there is duplicate indexes?

In [9]:
s1 = pd.Series([0, 1], index=['a', 'b'])
s2 = pd.Series([2, 3, 4], index=['c', 'd', 'b'])
s3 = pd.Series([5, 6], index=['f', 'g'])
pd.concat([s1, s2, s3])
# Index Repeats.

a    0
b    1
c    2
d    3
b    4
f    5
g    6
dtype: int64

By default `concat` works along `axis=0`, producing another Series. If you pass `axis=1`,
the result will instead be a DataFrame (`axis=1` is the columns):

In [12]:
s1 = pd.Series([0, 1], index=['a', 'b'])
s2 = pd.Series([2, 3, 4], index=['c', 'd', 'e'])
s3 = pd.Series([5, 6], index=['f', 'g'])
pd.concat([s1, s2, s3], axis=1)

,0,1,2
a,0.0,NaN,NaN
b,1.0,NaN,NaN
c,NaN,2.0,NaN
d,NaN,3.0,NaN
e,NaN,4.0,NaN
f,NaN,NaN,5.0
g,NaN,NaN,6.0


In this case there is no overlap on the other axis, which as you can see is the sorted
union (the `'outer'` join) of the indexes. You can instead intersect them by passing
`join='inner'`:

In [13]:
s4 = pd.concat([s1, s3])

In [14]:
s4

a    0
b    1
f    5
g    6
dtype: int64

In [16]:
s1

a    0
b    1
dtype: int64

In [18]:
pd.concat([s1, s4], axis=1)

,0,1
a,0.0,0
b,1.0,1
f,NaN,5
g,NaN,6


In [17]:
pd.concat([s1, s4], axis=1, join='inner')

,0,1
a,0,0
b,1,1


In this last example, the `'f'` and `'g'` labels disappeared because of the `join='inner'`
option.

-------

##### ✅ Why column **0** has decimal (float) values after `concat`

This is a **very important Pandas concept → dtype up-casting due to missing values (NaN).**

Let’s understand step-by-step like a data science architect 👇

---

##### 🔹 Step 1 — Your original Series

```
s1
a    0
b    1
dtype: int64
```

```
s4
a    0
b    1
f    5
g    6
dtype: int64
```

Both are **integer series (`int64`)**

---

#####🔹 Step 2 — Column wise concat

```
pd.concat([s1, s4], axis=1)
```

This means:

👉 Pandas must **align on index labels**

Final index becomes **union of both indexes**

```
a, b, f, g
```

Now Pandas constructs the table:

| index | s1  | s4 |
| ----- | --- | -- |
| a     | 0   | 0  |
| b     | 1   | 1  |
| f     | ??? | 5  |
| g     | ??? | 6  |

For `f` and `g` → `s1` has **no values**

So Pandas inserts:

```
NaN
```

---

##### 🔹 Step 3 — Why integers become float

This is the key rule:

> **NumPy integer dtype CANNOT store NaN**

NaN is a **floating point concept**

Therefore Pandas must **upcast entire column to float**

So column 0 becomes:

```
[0.0, 1.0, NaN, NaN]
```

dtype becomes:

```
float64
```

---

##### 🔹 Deep intuition (VERY important for DS)

Think:

| dtype   | Can store NaN ? |
| ------- | --------------- |
| int64   | ❌ No            |
| float64 | ✅ Yes           |

So whenever Pandas introduces missing values →

👉 integer columns automatically become float

This happens in:

* concat
* merge
* reindex
* align
* join
* pivot

This is one of the **most common beginner confusions in Pandas**

---

#####🔹 How to prevent float conversion (modern solution)

Use **nullable integer dtype**

```
pd.concat([s1.astype("Int64"), s4.astype("Int64")], axis=1)
```

Now result:

```
      0     1
a     0     0
b     1     1
f   <NA>    5
g   <NA>    6
```

dtype stays:

```
Int64
```

This is **very important in real data science pipelines**

---

##### 🔹 Senior Architect Tip ⭐

Whenever you see:

> integers suddenly becoming decimal

Immediately ask:

👉 **Did Pandas introduce missing values somewhere?**

99% of time → YES

---




s1

In [25]:
s4

a    0
b    1
f    5
g    6
dtype: int64

In [26]:
pd.concat([s1, s4], axis=1, join_axes=[['a', 'c', 'b', 'e']])

TypeError: concat() got an unexpected keyword argument 'join_axes'

##### ✅ Why you are getting this error

```
TypeError: concat() got an unexpected keyword argument 'join_axes'
```

Because **`join_axes` was removed from pandas** (deprecated long back → removed in newer versions).

So modern pandas simply does **not recognise this argument.**

---

##### 🔹 What `join_axes` used to do (OLD pandas)

Earlier you could force concat to use a **specific index**

Example (old behaviour):

```
pd.concat([s1, s4], axis=1, join_axes=[['a','c','b','e']])
```

Meaning:

👉 “After concat, use THIS index order only.”

---

##### 🔹 Modern Pandas Solution ✅ (Very Important)

Now you must use:

> **`reindex()` after concat**

---

##### ✔ Correct way

```
result = pd.concat([s1, s4], axis=1)
result = result.reindex(['a','c','b','e'])
```

Output:

```
     0    1
a  0.0  0.0
c  NaN  NaN
b  1.0  1.0
e  NaN  NaN
```

---

##### 🔹 Even Better (Architect Level Thinking ⭐)

You can control behaviour **during concat itself** using:

##### ✅ 1. `join='inner'`

Keeps only common index

```
pd.concat([s1, s4], axis=1, join='inner')
```

Result:

```
   0  1
a  0  0
b  1  1
```

---

##### ✅ 2. `join='outer'` (default)

Keeps union of index

```
pd.concat([s1, s4], axis=1, join='outer')
```

---

##### 🔹 SUPER IMPORTANT Concept (Data Science Pipeline Insight)

Think of concat in 3 stages:

##### Stage 1 → Alignment Rule

(inner / outer)

##### Stage 2 → Build Result Index

##### Stage 3 → If you want custom index → `reindex()`

So **modern pandas philosophy is:**

> concat = combine
> reindex = shape control

Earlier pandas mixed both → now clean separation.

---

##### 🔹 Pro Tip ⭐ (Cleaner one-liner)

```
pd.concat([s1, s4], axis=1).reindex(['a','c','b','e'])
```

---


pd.concat([s1, s4], axis=1).reindex(['a','c','b','e'])


A potential issue is that the concatenated pieces are not identifiable in the result. Suppose
instead you wanted to create a hierarchical index on the concatenation axis. To
do this, use the `keys` argument:

In [28]:
result = pd.concat([s1, s1, s3], keys=['one', 'two', 'three'])

In [29]:
result

one    a    0
       b    1
two    a    0
       b    1
three  f    5
       g    6
dtype: int64

In [30]:
result.unstack()

,a,b,f,g
one,0.0,1.0,NaN,NaN
two,0.0,1.0,NaN,NaN
three,NaN,NaN,5.0,6.0


Note above that when you unstackked, due to NaN, the integer values become Decimal.

In the case of combining Series along `axis=1`, the `keys` become the DataFrame column
headers:

In [31]:
pd.concat([s1, s2, s3], axis=1, keys=['one', 'two', 'three'])

,one,two,three
a,0.0,NaN,NaN
b,1.0,NaN,NaN
c,NaN,2.0,NaN
d,NaN,3.0,NaN
e,NaN,4.0,NaN
f,NaN,NaN,5.0
g,NaN,NaN,6.0


-----

The same logic extends to DataFrame objects:

In [33]:
df1 = pd.DataFrame(np.arange(6).reshape(3, 2), index=['a', 'b', 'c'],columns=['one', 'two'])

In [34]:
df2 = pd.DataFrame(5 + np.arange(4).reshape(2, 2), index=['a', 'c'],columns=['three', 'four'])

In [35]:
df1

,one,two
a,0,1
b,2,3
c,4,5


In [36]:
df2

,three,four
a,5,6
c,7,8


In [37]:
pd.concat([df1, df2], axis=1, keys=['level1', 'level2'])

level1     level2     
     one two  three four
a      0   1    5.0  6.0
b      2   3    NaN  NaN
c      4   5    7.0  8.0

If you pass a dict of objects instead of a list, the dict’s keys will be used for the `keys`
option:

In [39]:
pd.concat({'level1': df1, 'level2': df2}, axis=1)

level1     level2     
     one two  three four
a      0   1    5.0  6.0
b      2   3    NaN  NaN
c      4   5    7.0  8.0

Above, two data frames aare passed as dictionary and then conctenated. 

---

There are additional arguments governing how the hierarchical index is created (see
Table 8-3). For example, we can name the created axis levels with the `names`
argument:

In [45]:
pd.concat([df1, df2], axis=1, keys=['level1', 'level2'],names=['upper', 'lower'])

upper level1     level2     
lower    one two  three four
a          0   1    5.0  6.0
b          2   3    NaN  NaN
c          4   5    7.0  8.0

In [46]:
columns = ["Argument", "Description"]

rows = [
["objs","List or dict of pandas objects to be concatenated; this is the only required argument"],
["axis","Axis to concatenate along; defaults to 0 (along rows)"],
["join","Either 'inner' or 'outer' ('outer' by default); intersection or union of indexes on other axes"],
["join_axes","Specific indexes to use for other axes instead of union/intersection logic"],
["keys","Values associated with objects being concatenated to form hierarchical index"],
["levels","Specific indexes to use as hierarchical index levels if keys passed"],
["names","Names for created hierarchical index levels"],
["verify_integrity","Check new axis for duplicates and raise exception if found"],
["ignore_index","Do not preserve original index; create new range index"]
]

render_book_table(
    "Table 8-3. concat Function Arguments",
    columns,
    rows
)

Argument,Description
objs,List or dict of pandas objects to be concatenated; this is the only required argument
axis,Axis to concatenate along; defaults to 0 (along rows)
join,Either 'inner' or 'outer' ('outer' by default); intersection or union of indexes on other axes
join_axes,Specific indexes to use for other axes instead of union/intersection logic
keys,Values associated with objects being concatenated to form hierarchical index
levels,Specific indexes to use as hierarchical index levels if keys passed
names,Names for created hierarchical index levels
verify_integrity,Check new axis for duplicates and raise exception if found
ignore_index,Do not preserve original index; create new range index


A last consideration concerns DataFrames in which the row index does not contain
any relevant data:

In [48]:
df1 = pd.DataFrame(np.random.randn(3, 4), columns=['a', 'b', 'c', 'd'])

In [49]:
df2 = pd.DataFrame(np.random.randn(2, 3), columns=['b', 'd', 'a'])

In [50]:
df1

,a,b,c,d
0,1.332600,-0.158554,-0.306327,0.853606
1,-1.435680,-1.106542,0.498967,0.211613
2,-0.209044,-1.217168,0.647688,1.507807


In [51]:
df2

,b,d,a
0,0.170734,-0.759426,-3.208594
1,0.299790,-0.276153,0.868087


In [52]:
pd.concat([df1, df2], ignore_index=True)

,a,b,c,d
0,1.332600,-0.158554,-0.306327,0.853606
1,-1.435680,-1.106542,0.498967,0.211613
2,-0.209044,-1.217168,0.647688,1.507807
3,-3.208594,0.170734,NaN,-0.759426
4,0.868087,0.299790,NaN,-0.276153


### Combining Data with Overlap

There is another data combination situation that can’t be expressed as either a merge
or concatenation operation. You may have two datasets whose indexes overlap in full
or part. As a motivating example, consider NumPy’s `where` function, which performs
the array-oriented equivalent of an `if-else` expression:

In [54]:
a = pd.Series([np.nan, 2.5, np.nan, 3.5, 4.5, np.nan],
              index=['f', 'e', 'd', 'c', 'b', 'a'])

In [56]:
b = pd.Series(np.arange(len(a), dtype=np.float64),
              index=['f', 'e', 'd', 'c', 'b', 'a'])

In [57]:
b[-1] = np.nan

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_17096\1621735104.py:1: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  b[-1] = np.nan


In [58]:
a

f    NaN
e    2.5
d    NaN
c    3.5
b    4.5
a    NaN
dtype: float64

In [59]:
b

f    0.0
e    1.0
d    2.0
c    3.0
b    4.0
a    NaN
dtype: float64

In [60]:
np.where(pd.isnull(a), b, a)

array([0. , 2.5, 2. , 3.5, 4.5, nan])

`'where'` function applies the condition in the first argument and then puts value from second or third argument in final data.

---

Series has a `combine_first` method, which performs the equivalent of this operation
along with pandas’s usual data alignment logic:

In [66]:
b

f    0.0
e    1.0
d    2.0
c    3.0
b    4.0
a    NaN
dtype: float64

In [64]:
b[:-2]

f    0.0
e    1.0
d    2.0
c    3.0
dtype: float64

In [65]:
a[2:]

d    NaN
c    3.5
b    4.5
a    NaN
dtype: float64

In [67]:
b[:-2].combine_first(a[2:])

a    NaN
b    4.5
c    3.0
d    2.0
e    1.0
f    0.0
dtype: float64

---

With DataFrames, `combine_first` does the same thing column by column, so you
can think of it as “patching” missing data in the calling object with data from the
object you pass:

In [80]:
df1 = pd.DataFrame({'a': [1., np.nan, 5., np.nan],
                    'b': [np.nan, 2., np.nan, 89.33],
                    'c': range(2, 18, 4)})

In [81]:
df2 = pd.DataFrame({'a': [5., 4., np.nan, 3., 7.],
                    'b': [np.nan, 3., 4., 99.99, 8.]})

In [82]:
df1

,a,b,c
0,1.0,NaN,2
1,NaN,2.00,6
2,5.0,NaN,10
3,NaN,89.33,14


In [83]:
df2

,a,b
0,5.0,NaN
1,4.0,3.00
2,NaN,4.00
3,3.0,99.99
4,7.0,8.00


In [84]:
df1.combine_first(df2)

,a,b,c
0,1.0,NaN,2.0
1,4.0,2.00,6.0
2,5.0,4.00,10.0
3,3.0,89.33,14.0
4,7.0,8.00,NaN


## 8.3 Reshaping and Pivoting

There are a number of basic operations for rearranging tabular data. These are alternatingly
referred to as reshape or pivot operations.

### Reshaping with Hierarchical Indexing

Hierarchical indexing provides a consistent way to rearrange data in a DataFrame.
There are two primary actions:

####  `stack`

> This “rotates” or pivots from the columns in the data to the rows

#### `unstack`

> This pivots from the rows into the columns

I’ll illustrate these operations through a series of examples. Consider a small Data‐
Frame with string arrays as row and column indexes:

In [86]:
data = pd.DataFrame(np.arange(6).reshape((2, 3)),
                    index=pd.Index(['Ohio', 'Colorado'], name='state'),
                    columns=pd.Index(['one', 'two', 'three'],name='number'))
                                     

In [87]:
data

number,one,two,three
state,,,
Ohio,0,1,2
Colorado,3,4,5


Using the `stack` method on this data pivots the columns into the rows, producing a
Series:

In [88]:
result = data.stack()

In [89]:
result

state     number
Ohio      one       0
          two       1
          three     2
Colorado  one       3
          two       4
          three     5
dtype: int64

From a hierarchically indexed Series, you can rearrange the data back into a Data‐
Frame with `unstack`:

In [90]:
result.unstack()

number,one,two,three
state,,,
Ohio,0,1,2
Colorado,3,4,5


By default the innermost level is unstacked (same with `stack`). You can unstack a different
level by passing a level number or name:

In [91]:
result.unstack(0)

state,Ohio,Colorado
number,,
one,0,3
two,1,4
three,2,5


In [92]:
result.unstack('state')

state,Ohio,Colorado
number,,
one,0,3
two,1,4
three,2,5


---

Unstacking might introduce missing data if all of the values in the level aren’t found
in each of the subgroups:

In [93]:
s1 = pd.Series([0, 1, 2, 3], index=['a', 'b', 'c', 'd'])

In [94]:
s2 = pd.Series([4, 5, 6], index=['c', 'd', 'e'])

In [95]:
data2 = pd.concat([s1, s2], keys=['one', 'two'])

In [96]:
data2

one  a    0
     b    1
     c    2
     d    3
two  c    4
     d    5
     e    6
dtype: int64

In [97]:
data2.unstack()

,a,b,c,d,e
one,0.0,1.0,2.0,3.0,NaN
two,NaN,NaN,4.0,5.0,6.0


Stacking filters out missing data by default, so the operation is more easily invertible:

In [103]:
data2.unstack().stack()

one  a    0.0
     b    1.0
     c    2.0
     d    3.0
two  c    4.0
     d    5.0
     e    6.0
dtype: float64

When you unstack in a DataFrame, the level unstacked becomes the lowest level in
the result:

In [99]:
df = pd.DataFrame({'left': result, 'right': result + 5},columns=pd.Index(['left', 'right'], name='side'))

In [100]:
df

side             left  right
state    number             
Ohio     one        0      5
         two        1      6
         three      2      7
Colorado one        3      8
         two        4      9
         three      5     10

In [101]:
df.unstack('state')

side   left          right         
state  Ohio Colorado  Ohio Colorado
number                             
one       0        3     5        8
two       1        4     6        9
three     2        5     7       10

When calling stack, we can indicate the name of the axis to stack:

In [102]:
df.unstack('state').stack('side')

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_17096\3997798123.py:1: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df.unstack('state').stack('side')


state         Ohio  Colorado
number side                 
one    left      0         3
       right     5         8
two    left      1         4
       right     6         9
three  left      2         5
       right     7        10

-----

##### ✅ Why `data2.unstack().stack()` Did NOT Warn but `df.unstack('state').stack('side')` Did

* `data2.unstack().stack()` is a **simple reshape**

  * No named level selection
  * No complex MultiIndex columns
  * No large missing-combination grid
    → Pandas uses a **straightforward stack path → no deprecation warning**

* `df.unstack('state').stack('side')` is a **more complex reshape**

  * You are stacking a **specific named level (`side`)**
  * Columns are now a **MultiIndex after unstack**
  * Pandas must choose between **old vs new stack engine**
    → So it shows a **FutureWarning (not an error)** saying old behaviour will be removed

To silence:

```python
df.unstack('state').stack('side', future_stack=True)
```

---

##### ⭐ Why Pandas Introduced a New `stack()` Implementation

Old stack had issues:

* Silent **sorting of levels**
* Automatic **dropping of missing combinations**
* **Inconsistent reshape round-trip**
* Slower on large MultiIndex
* Behaviour depended on internal execution path

So pandas (from 2.1+) introduced a **new deterministic reshape engine**.

---

##### ⭐ Key Behaviour Differences — Old vs New Stack

###### 1️⃣ Ordering

**Old stack**

* May silently **sort index / column levels**

**New stack**

* **Preserves original order**
  → Important for joins, ML pipelines, feature matrices

---

###### 2️⃣ Missing Combination (NaN Grid) Handling

**Old stack**

* Drops missing pairs automatically
* Result becomes **structurally smaller**

**New stack**

* Keeps full combination grid (more faithful reshape)
* Better **reversibility and predictability**

---

###### 3️⃣ Round-Trip Reliability

**Old**

```
df → unstack → stack  ≠  original df  (sometimes)
```

**New**

```
df → unstack → stack(future_stack=True)  ≈  same structure
```

Very important in:

* panel data
* time-series reshaping
* feature engineering

---

###### 4️⃣ Performance

New implementation:

* More vectorized internally
* Fewer index reconstructions
* Faster for large hierarchical datasets

---

###### 5️⃣ Level Selection Semantics

When doing:

```python
df.stack('side')
```

Old behaviour could:

* infer levels loosely
* behave differently if ordering / naming changed

New behaviour:

* **Strict + predictable level resolution**

---

##### ⭐ Practical Modern Pandas Recommendation

Always prefer:

```python
df.stack(level='side', future_stack=True)
```

This makes your code:

* forward compatible
* deterministic
* safer for production data science workflows

---






#### Pivoting “Long” to “Wide” Format

A common way to store multiple time series in databases and CSV is in so-called long
or stacked format. Let’s load some example data and do a small amount of time series
wrangling and other data cleaning:

In [104]:
data = pd.read_csv('examples/macrodata.csv')

In [105]:
data

,year,quarter,realgdp,realcons,realinv,realgovt,realdpi,cpi,m1,tbilrate,unemp,pop,infl,realint
0,1959,1,2710.349,1707.4,286.898,470.045,1886.9,28.980,139.7,2.82,5.8,177.146,0.00,0.00
1,1959,2,2778.801,1733.7,310.859,481.301,1919.7,29.150,141.7,3.08,5.1,177.830,2.34,0.74
2,1959,3,2775.488,1751.8,289.226,491.260,1916.4,29.350,140.5,3.82,5.3,178.657,2.74,1.09
3,1959,4,2785.204,1753.7,299.356,484.052,1931.3,29.370,140.0,4.33,5.6,179.386,0.27,4.06
4,1960,1,2847.699,1770.5,331.722,462.199,1955.5,29.540,139.6,3.50,5.2,180.007,2.31,1.19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
198,2008,3,13324.600,9267.7,1990.693,991.551,9838.3,216.889,1474.7,1.17,6.0,305.270,-3.16,4.33
199,2008,4,13141.920,9195.3,1857.661,1007.273,9920.4,212.174,1576.5,0.12,6.9,305.952,-8.79,8.91
200,2009,1,12925.410,9209.2,1558.494,996.287,9926.4,212.671,1592.8,0.22,8.1,306.547,0.94,-0.71
201,2009,2,12901.504,9189.0,1456.678,1023.528,10077.5,214.469,1653.6,0.18,9.2,307.226,3.37,-3.19


In [106]:
periods = pd.PeriodIndex(year=data.year, quarter=data.quarter,name='date')

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_17096\4101540932.py:1: FutureWarning: Constructing PeriodIndex from fields is deprecated. Use PeriodIndex.from_fields instead.
  periods = pd.PeriodIndex(year=data.year, quarter=data.quarter,name='date')


In [107]:
periods

PeriodIndex(['1959Q1', '1959Q2', '1959Q3', '1959Q4', '1960Q1', '1960Q2',
             '1960Q3', '1960Q4', '1961Q1', '1961Q2',
             ...
             '2007Q2', '2007Q3', '2007Q4', '2008Q1', '2008Q2', '2008Q3',
             '2008Q4', '2009Q1', '2009Q2', '2009Q3'],
            dtype='period[Q-DEC]', name='date', length=203)

In [108]:
periods = pd.PeriodIndex.from_fields(
    year=data.year,
    quarter=data.quarter,
    name='date'
)

TypeError: PeriodIndex.from_fields() got an unexpected keyword argument 'name'

In [109]:
periods = pd.PeriodIndex.from_fields(
    year=data.year,
    quarter=data.quarter
).rename('date')

In [110]:
periods

PeriodIndex(['1959Q1', '1959Q2', '1959Q3', '1959Q4', '1960Q1', '1960Q2',
             '1960Q3', '1960Q4', '1961Q1', '1961Q2',
             ...
             '2007Q2', '2007Q3', '2007Q4', '2008Q1', '2008Q2', '2008Q3',
             '2008Q4', '2009Q1', '2009Q2', '2009Q3'],
            dtype='period[Q-DEC]', name='date', length=203)

 `pd.PeriodIndex.from_fields` makes a Index with Year and Q as special Data type  `'period[Q-DEC]'`

In [111]:
columns = pd.Index(['realgdp', 'infl', 'unemp'], name='item')

In [112]:
columns

Index(['realgdp', 'infl', 'unemp'], dtype='object', name='item')

In [113]:
data = data.reindex(columns=columns)

In [114]:
data

item,realgdp,infl,unemp
0,2710.349,0.00,5.8
1,2778.801,2.34,5.1
2,2775.488,2.74,5.3
3,2785.204,0.27,5.6
4,2847.699,2.31,5.2
...,...,...,...
198,13324.600,-3.16,6.0
199,13141.920,-8.79,6.9
200,12925.410,0.94,8.1
201,12901.504,3.37,9.2


Above step is just removing additional not needed columns from the data set.

In [115]:
data.index = periods.to_timestamp('D', 'end')

In [116]:
data

item,realgdp,infl,unemp
date,,,
1959-03-31 23:59:59.999999999,2710.349,0.00,5.8
1959-06-30 23:59:59.999999999,2778.801,2.34,5.1
1959-09-30 23:59:59.999999999,2775.488,2.74,5.3
1959-12-31 23:59:59.999999999,2785.204,0.27,5.6
1960-03-31 23:59:59.999999999,2847.699,2.31,5.2
...,...,...,...
2008-09-30 23:59:59.999999999,13324.600,-3.16,6.0
2008-12-31 23:59:59.999999999,13141.920,-8.79,6.9
2009-03-31 23:59:59.999999999,12925.410,0.94,8.1


In above step we added the `periods` object's `'period[Q-DEC]'` index as index for the data, but with format D , which means last sec of last day of the Q

In [128]:
ldata  = data.stack().reset_index().rename(columns={0: 'value'})

In [129]:
ldata

,date,item,value
0,1959-03-31 23:59:59.999999999,realgdp,2710.349
1,1959-03-31 23:59:59.999999999,infl,0.000
2,1959-03-31 23:59:59.999999999,unemp,5.800
3,1959-06-30 23:59:59.999999999,realgdp,2778.801
4,1959-06-30 23:59:59.999999999,infl,2.340
...,...,...,...
604,2009-06-30 23:59:59.999999999,infl,3.370
605,2009-06-30 23:59:59.999999999,unemp,9.200
606,2009-09-30 23:59:59.999999999,realgdp,12990.341
607,2009-09-30 23:59:59.999999999,infl,3.560


----
Thisis how ldata was formed.

In [122]:
ldata_temp1=data.stack()

In [123]:
ldata_temp1

date                           item   
1959-03-31 23:59:59.999999999  realgdp     2710.349
                               infl           0.000
                               unemp          5.800
1959-06-30 23:59:59.999999999  realgdp     2778.801
                               infl           2.340
                                            ...    
2009-06-30 23:59:59.999999999  infl           3.370
                               unemp          9.200
2009-09-30 23:59:59.999999999  realgdp    12990.341
                               infl           3.560
                               unemp          9.600
Length: 609, dtype: float64

In [124]:
ldata_temp2=ldata_temp1.reset_index()

In [125]:
ldata_temp2

,date,item,0
0,1959-03-31 23:59:59.999999999,realgdp,2710.349
1,1959-03-31 23:59:59.999999999,infl,0.000
2,1959-03-31 23:59:59.999999999,unemp,5.800
3,1959-06-30 23:59:59.999999999,realgdp,2778.801
4,1959-06-30 23:59:59.999999999,infl,2.340
...,...,...,...
604,2009-06-30 23:59:59.999999999,infl,3.370
605,2009-06-30 23:59:59.999999999,unemp,9.200
606,2009-09-30 23:59:59.999999999,realgdp,12990.341
607,2009-09-30 23:59:59.999999999,infl,3.560


In [126]:
ldata_temp3=ldata_temp2.rename(columns={0: 'value'})

In [127]:
ldata_temp3

,date,item,value
0,1959-03-31 23:59:59.999999999,realgdp,2710.349
1,1959-03-31 23:59:59.999999999,infl,0.000
2,1959-03-31 23:59:59.999999999,unemp,5.800
3,1959-06-30 23:59:59.999999999,realgdp,2778.801
4,1959-06-30 23:59:59.999999999,infl,2.340
...,...,...,...
604,2009-06-30 23:59:59.999999999,infl,3.370
605,2009-06-30 23:59:59.999999999,unemp,9.200
606,2009-09-30 23:59:59.999999999,realgdp,12990.341
607,2009-09-30 23:59:59.999999999,infl,3.560


------

This `ldata` is the so-called long format for multiple time series, or other observational data
with two or more keys (here, our keys are date and item). Each row in the table represents
a single observation.

Data is frequently stored this way in relational databases like MySQL, as a fixed
schema (column names and data types) allows the number of distinct values in the
`item` column to change as data is added to the table. In the previous example, `date`
and `item` would usually be the primary keys (in relational database parlance), offering
both relational integrity and easier joins. In some cases, the data may be more difficult
to work with in this format; you might prefer to have a DataFrame containing
one column per distinct item value indexed by timestamps in the date column. Data‐
Frame’s pivot method performs exactly this transformation:

In [130]:
pivoted = ldata.pivot('date', 'item', 'value')

TypeError: DataFrame.pivot() takes 1 positional argument but 4 were given

In [131]:
pivoted = ldata.pivot(
    index='date',
    columns='item',
    values='value'
)

item,infl,realgdp,unemp
date,,,
1959-03-31 23:59:59.999999999,0.00,2710.349,5.8
1959-06-30 23:59:59.999999999,2.34,2778.801,5.1
1959-09-30 23:59:59.999999999,2.74,2775.488,5.3
1959-12-31 23:59:59.999999999,0.27,2785.204,5.6
1960-03-31 23:59:59.999999999,2.31,2847.699,5.2
...,...,...,...
2008-09-30 23:59:59.999999999,-3.16,13324.600,6.0
2008-12-31 23:59:59.999999999,-8.79,13141.920,6.9
2009-03-31 23:59:59.999999999,0.94,12925.410,8.1


✅ Why This Error Happened

You wrote:

```python
pivoted = ldata.pivot('date', 'item', 'value')
```

And pandas gave:

```
TypeError: DataFrame.pivot() takes 1 positional argument but 4 were given
```

This means:

👉 **New pandas versions no longer allow positional arguments in `pivot()`**

Earlier (very old pandas / old books like Wes McKinney DS book):

```python
df.pivot('row', 'column', 'value')
```

was allowed.

Now pandas requires **keyword arguments only.**

---

##### ⭐ Correct Modern Syntax

You must write:

```python
pivoted = ldata.pivot(
    index='date',
    columns='item',
    values='value'
)
```

This will work.

---

##### ⭐ Why Pandas Changed This

Pandas developers are making APIs:

* more explicit
* less error-prone
* easier to read
* easier to maintain

Because positional style caused confusion:

Example ambiguity:

```python
df.pivot('A','B','C')
```

Reader must remember:

* which is index?
* which is columns?
* which is values?

Now:

```python
df.pivot(index='A', columns='B', values='C')
```

✔ self-documenting
✔ safer in production DS code

---

##### ⭐ VERY Important Concept (Senior DS Insight)

`pivot()` is basically:

> **Reshape long → wide**

So your data:

| date | item | value |
| ---- | ---- | ----- |

becomes:

| date | item1 | item2 | item3 |

This is extremely common in:

* time series feature engineering
* ML matrix building
* financial panel reshaping

---

##### ⭐ Also Know This (Interview Level)

If duplicates exist:

```python
pivot() → ERROR
pivot_table() → works (aggregation)
```

Example:

```python
ldata.pivot_table(
    index='date',
    columns='item',
    values='value'
)
```

---

##### ✅ Final Fix

Just change to:

```python
pivoted = ldata.pivot(
    index='date',
    columns='item',
    values='value'
)
```

---



The first two values passed are the columns to be used respectively as the row and
column index, then finally an optional value column to fill the DataFrame. Suppose
you had two value columns that you wanted to reshape simultaneously:

In [134]:
ldata['value2'] = np.random.randn(len(ldata))

In [135]:
ldata

,date,item,value,value2
0,1959-03-31 23:59:59.999999999,realgdp,2710.349,0.479259
1,1959-03-31 23:59:59.999999999,infl,0.000,-1.850061
2,1959-03-31 23:59:59.999999999,unemp,5.800,1.187355
3,1959-06-30 23:59:59.999999999,realgdp,2778.801,-0.217157
4,1959-06-30 23:59:59.999999999,infl,2.340,0.257260
...,...,...,...,...
604,2009-06-30 23:59:59.999999999,infl,3.370,-0.332260
605,2009-06-30 23:59:59.999999999,unemp,9.200,2.986624
606,2009-09-30 23:59:59.999999999,realgdp,12990.341,-0.799127
607,2009-09-30 23:59:59.999999999,infl,3.560,0.619767


By omitting the last argument, you obtain a DataFrame with hierarchical columns:

In [137]:
pivoted = ldata.pivot(
    index='date',
    columns='item',
)

In [138]:
pivoted

value                     value2            \
item                           infl    realgdp unemp      infl   realgdp   
date                                                                       
1959-03-31 23:59:59.999999999  0.00   2710.349   5.8 -1.850061  0.479259   
1959-06-30 23:59:59.999999999  2.34   2778.801   5.1  0.257260 -0.217157   
1959-09-30 23:59:59.999999999  2.74   2775.488   5.3  0.626444 -0.039666   
1959-12-31 23:59:59.999999999  0.27   2785.204   5.6  2.338407 -0.145449   
1960-03-31 23:59:59.999999999  2.31   2847.699   5.2 -1.613560 -0.839448   
...                             ...        ...   ...       ...       ...   
2008-09-30 23:59:59.999999999 -3.16  13324.600   6.0  0.450468  1.418511   
2008-12-31 23:59:59.999999999 -8.79  13141.920   6.9 -0.967712 -0.805432   
2009-03-31 23:59:59.999999999  0.94  12925.410   8.1 -0.330363 -1.957744   
2009-06-30 23:59:59.999999999  3.37  12901.504   9.2 -0.332260  0.412599   
2009-09-30 23:59:59.999999999  3.56  12990.341   9.6  0.619767 -0.799127   

                                         
item                              unemp  
date                                     
1959-03-31 23:59:59.999999999  1.187355  
1959-06-30 23:59:59.999999999 -0.360906  
1959-09-30 23:59:59.999999999 -1.563932  
1959-12-31 23:59:59.999999999 -0.634181  
1960-03-31 23:59:59.999999999 -1.414451  
...                                 ...  
2008-09-30 23:59:59.999999999  0.227950  
2008-12-31 23:59:59.999999999  0.281496  
2009-03-31 23:59:59.999999999 -0.399893  
2009-06-30 23:59:59.999999999  2.986624  
2009-09-30 23:59:59.999999999 -0.038745  

[203 rows x 6 columns]

In [139]:
pivoted['value'][:5]

item,infl,realgdp,unemp
date,,,
1959-03-31 23:59:59.999999999,0.00,2710.349,5.8
1959-06-30 23:59:59.999999999,2.34,2778.801,5.1
1959-09-30 23:59:59.999999999,2.74,2775.488,5.3
1959-12-31 23:59:59.999999999,0.27,2785.204,5.6
1960-03-31 23:59:59.999999999,2.31,2847.699,5.2


Note that `pivot` is equivalent to creating a hierarchical index using set_index followed
by a call to `unstack`:

In [141]:
unstacked = ldata.set_index(['date', 'item']).unstack('item')

In [142]:
unstacked

value                     value2            \
item                           infl    realgdp unemp      infl   realgdp   
date                                                                       
1959-03-31 23:59:59.999999999  0.00   2710.349   5.8 -1.850061  0.479259   
1959-06-30 23:59:59.999999999  2.34   2778.801   5.1  0.257260 -0.217157   
1959-09-30 23:59:59.999999999  2.74   2775.488   5.3  0.626444 -0.039666   
1959-12-31 23:59:59.999999999  0.27   2785.204   5.6  2.338407 -0.145449   
1960-03-31 23:59:59.999999999  2.31   2847.699   5.2 -1.613560 -0.839448   
...                             ...        ...   ...       ...       ...   
2008-09-30 23:59:59.999999999 -3.16  13324.600   6.0  0.450468  1.418511   
2008-12-31 23:59:59.999999999 -8.79  13141.920   6.9 -0.967712 -0.805432   
2009-03-31 23:59:59.999999999  0.94  12925.410   8.1 -0.330363 -1.957744   
2009-06-30 23:59:59.999999999  3.37  12901.504   9.2 -0.332260  0.412599   
2009-09-30 23:59:59.999999999  3.56  12990.341   9.6  0.619767 -0.799127   

                                         
item                              unemp  
date                                     
1959-03-31 23:59:59.999999999  1.187355  
1959-06-30 23:59:59.999999999 -0.360906  
1959-09-30 23:59:59.999999999 -1.563932  
1959-12-31 23:59:59.999999999 -0.634181  
1960-03-31 23:59:59.999999999 -1.414451  
...                                 ...  
2008-09-30 23:59:59.999999999  0.227950  
2008-12-31 23:59:59.999999999  0.281496  
2009-03-31 23:59:59.999999999 -0.399893  
2009-06-30 23:59:59.999999999  2.986624  
2009-09-30 23:59:59.999999999 -0.038745  

[203 rows x 6 columns]

In [144]:
ldata

,date,item,value,value2
0,1959-03-31 23:59:59.999999999,realgdp,2710.349,0.479259
1,1959-03-31 23:59:59.999999999,infl,0.000,-1.850061
2,1959-03-31 23:59:59.999999999,unemp,5.800,1.187355
3,1959-06-30 23:59:59.999999999,realgdp,2778.801,-0.217157
4,1959-06-30 23:59:59.999999999,infl,2.340,0.257260
...,...,...,...,...
604,2009-06-30 23:59:59.999999999,infl,3.370,-0.332260
605,2009-06-30 23:59:59.999999999,unemp,9.200,2.986624
606,2009-09-30 23:59:59.999999999,realgdp,12990.341,-0.799127
607,2009-09-30 23:59:59.999999999,infl,3.560,0.619767


In [145]:
ldata.set_index(['date', 'item'])

value    value2
date                          item                        
1959-03-31 23:59:59.999999999 realgdp   2710.349  0.479259
                              infl         0.000 -1.850061
                              unemp        5.800  1.187355
1959-06-30 23:59:59.999999999 realgdp   2778.801 -0.217157
                              infl         2.340  0.257260
...                                          ...       ...
2009-06-30 23:59:59.999999999 infl         3.370 -0.332260
                              unemp        9.200  2.986624
2009-09-30 23:59:59.999999999 realgdp  12990.341 -0.799127
                              infl         3.560  0.619767
                              unemp        9.600 -0.038745

[609 rows x 2 columns]

In [146]:
ldata.set_index(['date', 'item']).unstack('item')

value                     value2            \
item                           infl    realgdp unemp      infl   realgdp   
date                                                                       
1959-03-31 23:59:59.999999999  0.00   2710.349   5.8 -1.850061  0.479259   
1959-06-30 23:59:59.999999999  2.34   2778.801   5.1  0.257260 -0.217157   
1959-09-30 23:59:59.999999999  2.74   2775.488   5.3  0.626444 -0.039666   
1959-12-31 23:59:59.999999999  0.27   2785.204   5.6  2.338407 -0.145449   
1960-03-31 23:59:59.999999999  2.31   2847.699   5.2 -1.613560 -0.839448   
...                             ...        ...   ...       ...       ...   
2008-09-30 23:59:59.999999999 -3.16  13324.600   6.0  0.450468  1.418511   
2008-12-31 23:59:59.999999999 -8.79  13141.920   6.9 -0.967712 -0.805432   
2009-03-31 23:59:59.999999999  0.94  12925.410   8.1 -0.330363 -1.957744   
2009-06-30 23:59:59.999999999  3.37  12901.504   9.2 -0.332260  0.412599   
2009-09-30 23:59:59.999999999  3.56  12990.341   9.6  0.619767 -0.799127   

                                         
item                              unemp  
date                                     
1959-03-31 23:59:59.999999999  1.187355  
1959-06-30 23:59:59.999999999 -0.360906  
1959-09-30 23:59:59.999999999 -1.563932  
1959-12-31 23:59:59.999999999 -0.634181  
1960-03-31 23:59:59.999999999 -1.414451  
...                                 ...  
2008-09-30 23:59:59.999999999  0.227950  
2008-12-31 23:59:59.999999999  0.281496  
2009-03-31 23:59:59.999999999 -0.399893  
2009-06-30 23:59:59.999999999  2.986624  
2009-09-30 23:59:59.999999999 -0.038745  

[203 rows x 6 columns]

#### Pivoting “Wide” to “Long” Format

An inverse operation to `pivot` for DataFrames is `pandas.melt`. Rather than transforming
one column into many in a new DataFrame, it merges multiple columns into
one, producing a DataFrame that is longer than the input. Let’s look at an example:

In [155]:
df = pd.DataFrame({'key': ['foo', 'bar', 'baz'],
                   'A': [1, 2, 3],
                   'B': [4, 5, 6],
                   'C': [7, 8, 9]})

In [156]:
df

,key,A,B,C
0,foo,1,4,7
1,bar,2,5,8
2,baz,3,6,9


The `'key'` column may be a group indicator, and the other columns are data values.
When using `pandas.melt`, we must indicate which columns (if any) are group indicators.
Let’s use `'key'` as the only group indicator here:



In [157]:
melted = pd.melt(df, ['key'])

In [158]:
melted

,key,variable,value
0,foo,A,1
1,bar,A,2
2,baz,A,3
3,foo,B,4
4,bar,B,5
5,baz,B,6
6,foo,C,7
7,bar,C,8
8,baz,C,9


Using `pivot`, we can reshape back to the original layout:

In [165]:
reshaped = melted.pivot(index ='key', columns='variable', values= 'value')

In [166]:
reshaped

variable,A,B,C
key,,,
bar,2,5,8
baz,3,6,9
foo,1,4,7


Since the result of `pivot` creates an index from the column used as the row labels, we
may want to use `reset_index` to move the data back into a column:

In [170]:
reshaped.reset_index()

variable,key,A,B,C
0,bar,2,5,8
1,baz,3,6,9
2,foo,1,4,7


In [171]:
df

,key,A,B,C
0,foo,1,4,7
1,bar,2,5,8
2,baz,3,6,9


--- see how original df and final reshaped with reset index become identical

---

You can also specify a subset of columns to use as value columns:

In [175]:
pd.melt(df, id_vars=['key'], value_vars=['A', 'B'])

,key,variable,value
0,foo,A,1
1,bar,A,2
2,baz,A,3
3,foo,B,4
4,bar,B,5
5,baz,B,6


`pandas.melt` can be used without any group identifiers, too:



In [178]:
pd.melt(df, value_vars=['A', 'B', 'C'])

,variable,value
0,A,1
1,A,2
2,A,3
3,B,4
4,B,5
5,B,6
6,C,7
7,C,8
8,C,9


In [181]:
pd.melt(df, value_vars=['A', 'B', 'key'])

,variable,value
0,A,1
1,A,2
2,A,3
3,B,4
4,B,5
5,B,6
6,key,foo
7,key,bar
8,key,baz


We cna have more than one id vars also.

In [180]:
pd.melt(df, id_vars=['key', 'A'], value_vars=[ 'B'])

,key,A,variable,value
0,foo,1,B,4
1,bar,2,B,5
2,baz,3,B,6


## 8.4 Conclusion

Now that you have some pandas basics for data import, cleaning, and reorganization
under your belt, we are ready to move on to data visualization with matplotlib. We
will return to pandas later in the book when we discuss more advanced analytics.